# Size *within* each work-type archetype — CSI's two-axis model  *(Python)*
Twin of `scripts/2026-06-15_size_within_archetype.py`. A project = **work-type × size × messiness → profit**. 'Large' is a tier *inside* each work-type (native unit), not a type. Plan-knowable inputs only (7 quantities); `Project_Size_Dollars` is back-test only (leakage). Live `10_000` rule unchanged.

In [ ]:
from pathlib import Path
import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
warnings.filterwarnings('ignore'); pd.set_option('display.width',170); pd.set_option('display.max_columns',60)
BASE=Path('/Users/t/Library/Mobile Documents/com~apple~CloudDocs/00 MSBA + Coding/00_summer 2026/00_CSI_ELP/01_Summer_2026')
OUT=BASE/'02_modules/sweet_spot/outputs'
df=pd.read_csv(OUT/'2026-06-04_project_base.csv')
for c in ['MIX_TN_num','GRADE_CY_num','BASE_TN_num','SHLDR_TN_num','MILL_SY_num','Project_Size_Dollars','CSI_Margin_Pct','Num_Valid_Bidders']:
    df[c]=pd.to_numeric(df[c],errors='coerce')
df['placed_tons']=df[['MIX_TN_num','BASE_TN_num','SHLDR_TN_num']].fillna(0).sum(axis=1)
df['base_shldr']=df[['BASE_TN_num','SHLDR_TN_num']].fillna(0).sum(axis=1)
print(df.shape)

### Axis 1 — work-type from which scopes are present (plan-knowable)

In [ ]:
def work_type(r):  # canonical — split on grading = rebuild vs resurface
    mix,grade,mill=r['MIX_TN_num']>0,r['GRADE_CY_num']>0,r['MILL_SY_num']>0
    base,shldr=r['BASE_TN_num']>0,r['SHLDR_TN_num']>0
    if not r['has_quantity']: return 'Unknown/NoQty'
    if grade and mix: return 'Reconstruct/Rebuild'
    if mix and mill: return 'Mill & Overlay'
    if mix: return 'HMA Overlay/Resurface'
    if grade: return 'Grading/Earthwork'
    if base or shldr: return 'Aggregate/Base/Gravel'
    return 'Other'
df['work_type']=df.apply(work_type,axis=1)
df['is_bid']=df['CSI_Participated'].astype(str).str.upper()=='TRUE'
df['is_win']=df['CSI_Won'].astype(str).str.upper()=='TRUE'
df['is_win_well']=df['CSI_Bid_Quality'].isin({'Excellent Win','Acceptable Win'})
df['is_bomb']=df['CSI_Bid_Quality']=='Bomb'
print(df.loc[df.has_quantity==True,'work_type'].value_counts().to_string())

### Axis 3 — size cut derived *within* each work-type (native unit, GMM 2-mode seam else median)

In [ ]:
SIZE_BASIS={'Reconstruct/Rebuild':('placed_tons','placed tons'),
 'Mill & Overlay':('MIX_TN_num','MIX tons'),'HMA Overlay/Resurface':('MIX_TN_num','MIX tons'),
 'Grading/Earthwork':('GRADE_CY_num','GRADE CY'),'Aggregate/Base/Gravel':('base_shldr','BASE+SHLDR tons')}
def size_cut(v):
    v=np.asarray(v,float); v=v[v>0]
    if len(v)<30: return float(np.median(v)) if len(v) else np.nan
    x=np.log10(v).reshape(-1,1); lo,hi_p=np.percentile(v,[10,90])
    g1=GaussianMixture(1,random_state=42).fit(x); g2=GaussianMixture(2,random_state=42,n_init=5).fit(x)
    if g2.bic(x)<g1.bic(x):
        grid=np.linspace(x.min(),x.max(),4000).reshape(-1,1); hi=int(np.argmax(g2.means_.ravel()))
        sw=np.where(g2.predict(grid)==hi)[0]
        if len(sw):
            cut=float(10**grid[sw[0],0])
            if lo<=cut<=hi_p: return cut
    return float(np.median(v))
df['size_value']=np.nan; df['size_tier']=pd.NA; cuts={}
for wt,(basis,lab) in SIZE_BASIS.items():
    m=df['work_type']==wt; cut=size_cut(df.loc[m,basis]); cuts[wt]=cut
    df.loc[m,'size_value']=df.loc[m,basis]; df.loc[m,'size_tier']=np.where(df.loc[m,basis]>=cut,'Large','Small')
    print(f'{wt:24s} {lab:18s} Large >= {cut:>10,.0f}')

### Back-test (NOT an input): does within-type size rank-track realized $?

In [ ]:
for wt in SIZE_BASIS:
    s=df[(df.work_type==wt)&(df.Project_Size_Dollars>0)&(df.size_value>0)]
    if len(s)>10: print(f'{wt:24s} rho={s.size_value.corr(s.Project_Size_Dollars,method="spearman"):+.2f}')

### Sweet spot — profit by work-type × size (the 'where do we make money' table + heatmap)

In [ ]:
bid=df[df.is_bid & df.work_type.isin(SIZE_BASIS)].copy()
g=bid.groupby(['work_type','size_tier'],observed=True).agg(n_bid=('Contract_Id','size'),
  win_rate=('is_win',lambda s:round(s.mean()*100,1)), win_well=('is_win_well',lambda s:round(s.mean()*100,1)),
  bomb=('is_bomb',lambda s:round(s.mean()*100,1)), med_margin=('CSI_Margin_Pct',lambda s:round(s.median()*100,1))).reset_index()
print(g.to_string(index=False))
piv=g.pivot(index='work_type',columns='size_tier',values='med_margin')
fig,ax=plt.subplots(figsize=(6,4)); im=ax.imshow(piv.values,cmap='RdYlGn',aspect='auto',vmin=0,vmax=30)
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns); ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v=piv.values[i,j]
        if v==v: ax.text(j,i,f'{v:.0f}%',ha='center',va='center',fontsize=10)
ax.set_title('Median CSI margin by work-type × size'); fig.colorbar(im,label='margin %'); plt.tight_layout(); plt.show()

**Read:** smaller jobs within a type carry **higher margin** (more self-performed); Mill & Overlay is the commodity floor; big Reconstructs win most but at thin margin (keep-crews-busy, not margin). Matches the kickoff transcript.

### Forward classify a NEW plan (plan-knowable inputs only — leakage-safe)

In [ ]:
def classify(q):
    r={k:float(q.get(k,0) or 0) for k in ['MIX_TN_num','GRADE_CY_num','BASE_TN_num','SHLDR_TN_num','MILL_SY_num']}
    r['has_quantity']=q.get('has_quantity',any(v>0 for v in r.values())); wt=work_type(r)
    basis=SIZE_BASIS.get(wt,(None,None))[0]
    val={'placed_tons':r['MIX_TN_num']+r['BASE_TN_num']+r['SHLDR_TN_num'],'base_shldr':r['BASE_TN_num']+r['SHLDR_TN_num']}.get(basis, r.get(basis) if basis else np.nan)
    tier=('Large' if basis and val>=cuts.get(wt,np.inf) else 'Small') if basis else 'n/a'
    return {'work_type':wt,'size_value':round(val) if val==val else None,'size_tier':tier}
# Stutsman County grade-raise (your image): 48,442 CY borrow, no asphalt
print(classify({'GRADE_CY_num':48442,'BASE_TN_num':6546}))

## Takeaway
'Large' is now a **size tier within each work-type**, not a standalone archetype — and the sweet spot is **profit by work-type × size**. Client artifact: `2026-06-15_archetype_decision_rules_PROPOSED.md`.